# 01 - Data Audit
Scan all videos in the Google Drive folder and generate a metadata inventory.

In [2]:
# ===== CONFIGURATION =====
# GitHub -- do not change
GITHUB_REPO_URL = "https://github.com/kaarthik-balakrishnan/LightningPoseTrack.git"
GIT_BRANCH = "main"

# Google Drive root -- change this to match your Drive structure
DRIVE_ROOT = "/content/drive/MyDrive/PigBehavior/raw_videos"  # <-- SET THIS to your root
OUTPUT_ROOT = "/content/drive/MyDrive/LightningPoseTrack"

# Derived paths (change if your folders are at custom locations)
DRIVE_RAW_VIDEOS = f"{DRIVE_ROOT}/20260825_Behaving"    # Input: session video folders
DRIVE_REPORTS = f"{OUTPUT_ROOT}/reports"           # Output: inventory reports

# Google Drive folder ID (for reference)
DRIVE_FOLDER_ID = "1X_41ZW3HfwVeft2lPld3XNqXsdxRDIwb"

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os, sys
REPO_DIR = "/content/LightningPoseTrack"
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

Cloning into '/content/LightningPoseTrack'...
remote: Enumerating objects: 1396, done.
remote: Counting objects: 100% (159/159), done.
remote: Compressing objects: 100% (126/126), done.
remote: Total 1396 (delta 105), reused 70 (delta 31), pack-reused 1237 (from 1)
Receiving objects: 100% (1396/1396), 345.49 MiB | 42.80 MiB/s, done.
Resolving deltas: 100% (279/279), done.
/content/LightningPoseTrack


In [ ]:
!apt-get install -y -qq tesseract-ocr > /dev/null 2>&1
!pip install --quiet opencv-python pandas numpy pyarrow pytesseract

In [4]:
from pathlib import Path
from src.io.video_inventory import scan_videos

# Check what's in the raw videos directory
import subprocess
result = subprocess.run(["ls", "-la", DRIVE_RAW_VIDEOS], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

total 1146871
-rw------- 1 root root      4096 Aug 25 04:51 ._133738-4.ASF
-rw------- 1 root root   1409536 Aug 25 04:37 133738-4.ASF
-rw------- 1 root root      4096 Aug 25 07:52 ._133738-4.mp4
-rw------- 1 root root   1602400 Aug 25 07:52 133738-4.mp4
-rw------- 1 root root  79857664 Aug 25 04:42 133740-3.ASF
-rw------- 1 root root      4096 Aug 25 07:55 ._133740-3.mp4
-rw------- 1 root root 112582762 Aug 25 07:54 133740-3.mp4
-rw------- 1 root root   6586880 Aug 25 04:38 133743-1.ASF
-rw------- 1 root root      4096 Aug 25 07:56 ._133743-1.mp4
-rw------- 1 root root   9574472 Aug 25 07:54 133743-1.mp4
-rw------- 1 root root  73205760 Aug 25 04:42 133743-2.ASF
-rw------- 1 root root      4096 Aug 25 07:57 ._133743-2.mp4
-rw------- 1 root root 109507847 Aug 25 07:56 133743-2.mp4
-rw------- 1 root root  48334848 Aug 25 04:42 133752-4.ASF
-rw------- 1 root root      4096 Aug 25 07:56 ._133752-4.mp4
-rw------- 1 root root  59291394 Aug 25 07:57 133752-4.mp4
-rw------- 1 root root   51778

In [5]:
df = scan_videos(DRIVE_RAW_VIDEOS)
print(f"Found {len(df)} video files across {df['session'].nunique()} sessions")
df.head(20)

  Root path: /content/drive/MyDrive/PigBehavior/raw_videos/20260825_Behaving
  Path exists: True
  First 20 entries: ['134239-3.ASF', '133916-1.mp4', '133820-1.ASF', '133743-1.ASF', '133743-2.mp4', '133752-4.ASF', '134239-3.mp4', '134251-2.ASF', '133743-2.ASF', '._133743-1.mp4', '._134613-3.mp4', '._133916-1.mp4', '._133820-1.mp4', '._133738-4.mp4', '._134419-1.mp4', '._134251-2.mp4', '._134412-2.mp4', '._133743-2.mp4', '._134614-1.mp4', '._134706-2.mp4']
  ._133738-4.ASF — FAILED (no video stream)
  ._133738-4.mp4 — FAILED (no video stream)
  ._133740-3.mp4 — FAILED (no video stream)
  ._133743-1.mp4 — FAILED (no video stream)
  ._133743-2.mp4 — FAILED (no video stream)
  ._133752-4.mp4 — FAILED (no video stream)
  ._133820-1.mp4 — FAILED (no video stream)
  ._133916-1.mp4 — FAILED (no video stream)
  ._134239-3.mp4 — FAILED (no video stream)
  ._134251-2.mp4 — FAILED (no video stream)
  ._134258-4.mp4 — FAILED (no video stream)
  ._134412-2.mp4 — FAILED (no video stream)
  ._134419-1

,filename,path,session,camera,fps,frame_count,width,height,duration_sec,duration_min,recording_time
0,133738-4.ASF,133738-4.ASF,20260825_Behaving,4,10.00,112,1920,1080,11.20,0.19,
1,133738-4.mp4,133738-4.mp4,20260825_Behaving,4,10.00,42,1920,1080,4.20,0.07,
2,133740-3.ASF,133740-3.ASF,20260825_Behaving,3,10.00,3001,1920,1080,300.10,5.00,
3,133740-3.mp4,133740-3.mp4,20260825_Behaving,3,10.00,2854,1920,1080,285.40,4.76,
4,133743-1.ASF,133743-1.ASF,20260825_Behaving,1,10.25,369,1920,1080,36.00,0.60,
5,133743-1.mp4,133743-1.mp4,20260825_Behaving,1,10.25,147,1920,1080,14.34,0.24,
6,133743-2.ASF,133743-2.ASF,20260825_Behaving,2,10.25,3094,1920,1080,301.85,5.03,
7,133743-2.mp4,133743-2.mp4,20260825_Behaving,2,10.25,2184,1920,1080,213.07,3.55,
8,133752-4.ASF,133752-4.ASF,20260825_Behaving,4,5.00,1501,1920,1080,300.20,5.00,
9,133752-4.mp4,133752-4.mp4,20260825_Behaving,4,5.00,1425,1920,1080,285.00,4.75,


In [6]:
print("=== Summary ===")
print(f"Total videos: {len(df)}")
print(f"Total duration: {df['duration_min'].sum():.1f} min")
print(f"Sessions: {sorted(df['session'].unique())}")
print(f"Cameras: {sorted(df['camera'].unique())}")
print(f"\nPer session:")
print(df.groupby('session').agg(
    videos=('filename', 'count'),
    duration_min=('duration_min', 'sum'),
    cameras=('camera', lambda x: sorted(x.unique()))
))

=== Summary ===
Total videos: 46
Total duration: 67.7 min
Sessions: ['20260825_Behaving']
Cameras: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Per session:
                   videos  duration_min       cameras
session                                              
20260825_Behaving      46         67.67  [1, 2, 3, 4]


In [7]:
output_dir = Path(DRIVE_REPORTS)
output_dir.mkdir(parents=True, exist_ok=True)
csv_path = output_dir / "video_inventory.csv"
df.to_csv(csv_path, index=False)
print(f"Saved inventory to {csv_path}")

Saved inventory to /content/drive/MyDrive/LightningPoseTrack/reports/video_inventory.csv
